# Data Wrangling Notebook

#### Overall Goal: Build node/adjacency matrix features and info (PyTorch Geometric or Spektral suggested)

### 1. Members Graph

- **Nodes**: member_ids from member-edges 

- **Node Attributes**: hometown, city, state, lat, lon from meta-members; event_id, group_id from rsvps; weight from member-to-group-edges (number of shared groups between members)

- **Edges**: connections between members

- **Edge Attributes**: weight from member-edges  (what is weight here? how many events + grps they share? how calculated?)

- **Note**: not a digraph

### 2. Groups Graph

- **Nodes**: group_id from meta-groups 

- **Node Attributes**: : group_name, num_members, category_id, category_name, organizer_id, group_urlname from meta-groups

- **Edges**: connections between members

- **Edge Attributes**: weight from group-edges (number of shared members between groups)

- **Note**: also not a digraph

### Imports

In [60]:
# General libraries
import pandas as pd
import numpy as np

# Graphing libraries
import networkx as nx
import torch
from torch_geometric.utils.convert import from_networkx
from torch_geometric.data import Data

In [ ]:
%%capture

%run data.ipynb

In [29]:
# Example

# PyG has a function to convert nx graphs into Data objects 
data = from_networkx(G_sample, group_node_attrs='')
print(data.x)
print(data.edge_index)

tensor([[0.0133],
        [0.0000],
        [0.0000],
        ...,
        [0.0000],
        [0.0000],
        [0.0000]])
tensor([[   0,    0,    0,  ..., 3838, 3839, 3840],
        [   1,    2,    3,  ..., 3507, 3507, 3507]])


### Building Graph 1: Members

In [37]:
# Drop member 239124265, as no metadata exits for it (confirmed in debugging not shown)
subset = subset[(subset['member1'] != 239124265) & (subset['member2'] != 239124265)]

mem_edges = mem_edges[(mem_edges['member1'] != 239124265) & (mem_edges['member2'] != 239124265)]

In [54]:
# Create dictionary of dictionaries
mem_dict = mem_df.to_dict(orient='index')

In [55]:
mem_df

,hometown,city,state,lat,lon
member_id,,,,,
2069,Brentwood,Brentwood,TN,36.00,-86.79
8386,Nashville,Nashville,TN,36.07,-86.78
9205,Brentwood,Brentwood,TN,36.00,-86.79
17903,Not provided,Nashville,TN,36.13,-86.80
20418,"Huntington, WV",Nashville,TN,36.17,-86.72
...,...,...,...,...,...
239513469,Not provided,Nashville,TN,36.09,-86.82
239515413,Not provided,La Vergne,TN,36.00,-86.57
239519977,Not provided,Nashville,TN,36.17,-86.78


In [56]:
# Build graph from subset for now
G_m = nx.from_pandas_edgelist(
    subset,
    source='member1',
    target='member2',
    edge_attr='weight',
    create_using=nx.Graph())
# Set attributes 
nx.set_node_attributes(G_m, mem_dict)

In [57]:
print(nx.get_node_attributes(G_m, 'hometown'))

{198737924: 'Not provided', 220654721: 'Not provided', 208201738: 'Not provided', 88664332: 'Nashville', 8640526: 'Not provided', 56356372: 'Lewisburg', 183880473: 'Not provided', 194617630: 'Not provided', 185626278: 'Gainesville, FL', 8809394: 'Austin', 193825984: 'Not provided', 145838912: 'New York', 182472899: 'Springfield ', 8998727: 'Not provided', 3036107: 'Not provided', 127853262: 'Hermitage, TN', 209148366: 'Not provided', 207093714: 'Not provided', 204969170: 'Not provided', 156443612: 'Not provided', 12916961: 'Not provided', 66842212: 'Nashville', 216446825: 'Not provided', 183097071: 'Not provided', 212660721: 'Not provided', 160810612: 'Nashville', 200806774: 'Not provided', 154764282: 'Not provided', 14497407: 'Chicago', 73498632: 'Not provided', 66999812: 'Nashville', 221126666: 'Not provided', 197747724: 'Not provided', 91563022: 'Indianapolis, IN', 10798095: 'Savannah', 201765390: 'Not provided', 204465678: 'Not provided', 207654954: 'Not provided', 188889135: 'Not 

In [58]:
# Check the first few nodes' attributes
for node, attrs in list(G_m.nodes(data=True))[:3]:
    print(node, attrs)

198737924 {'hometown': 'Not provided', 'city': 'Nashville', 'state': 'TN', 'lat': 36.17, 'lon': -86.78}
220654721 {'hometown': 'Not provided', 'city': 'Nashville', 'state': 'TN', 'lat': 36.16, 'lon': -86.78}
208201738 {'hometown': 'Not provided', 'city': 'Nashville', 'state': 'TN', 'lat': 36.17, 'lon': -86.72}


In [59]:
# PyG has a function to convert nx graphs into Data objects 
data = from_networkx(G_m, group_node_attrs='all')

# Can't convert the strings to Data objects and can't One-Hot Encode the strings given memory issues

#print(data.x)
#print(data.edge_index)


AttributeError: 'list' object has no attribute 'dim'

### Building Graph 2: Groups

In [53]:
groups_df

,group_id,num_members,category_id,organizer_id
0,339011,15838,23,4353803
1,19728145,1778,5,118484462
2,6335372,2869,32,108448302
3,10016242,1975,34,8111102
4,21174496,2782,31,184580248
...,...,...,...,...
597,23742545,139,9,226783032
598,20647425,97,21,173945092
599,22504351,32,32,6141229
600,21686664,118,34,126066112


In [45]:
# Create dictionary of dictionaries
groups_dict = groups_df.to_dict(orient='index')

In [52]:
groups_dict

{0: {'group_id': 339011,
  'num_members': 15838,
  'category_id': 23,
  'organizer_id': 4353803},
 1: {'group_id': 19728145,
  'num_members': 1778,
  'category_id': 5,
  'organizer_id': 118484462},
 2: {'group_id': 6335372,
  'num_members': 2869,
  'category_id': 32,
  'organizer_id': 108448302},
 3: {'group_id': 10016242,
  'num_members': 1975,
  'category_id': 34,
  'organizer_id': 8111102},
 4: {'group_id': 21174496,
  'num_members': 2782,
  'category_id': 31,
  'organizer_id': 184580248},
 5: {'group_id': 11077852,
  'num_members': 918,
  'category_id': 28,
  'organizer_id': 4765912},
 6: {'group_id': 22197221,
  'num_members': 1812,
  'category_id': 23,
  'organizer_id': 199336381},
 7: {'group_id': 1585196,
  'num_members': 4828,
  'category_id': 23,
  'organizer_id': 13537265},
 8: {'group_id': 526316,
  'num_members': 3472,
  'category_id': 5,
  'organizer_id': 12229328},
 9: {'group_id': 1763190,
  'num_members': 1563,
  'category_id': 32,
  'organizer_id': 9890725},
 10: {'gr

In [46]:
# Build graph
G_g = nx.from_pandas_edgelist(
    subset,
    source='member1',
    target='member2',
    edge_attr='weight',
    create_using=nx.Graph())
# Set attributes 
nx.set_node_attributes(G_g, groups_dict)

In [50]:
print(nx.get_node_attributes(G_g, 'group_id'))

{}


In [47]:
G_g

In [ ]:
# PyG has a function to convert nx graphs into Data objects 
data = from_networkx(G_g, group_node_attrs='')
print(data.x)
print(data.edge_index)

RuntimeError: torch.cat(): expected a non-empty list of Tensors